# Anisotropic stress and three-dimensional mixed spaces

Independent exact-field checks accompany five-level campaigns for anisotropic stress, MsHHO and interior/normal enrichment. These are formulation checks with declared meshes, rather than historical figure reproductions.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


In [ ]:
from pymhm import TriangleMesh, solve_elasticity_mixed
from examples import core_extension_data as data
with threadpool_limits(1):
    stress = solve_elasticity_mixed(TriangleMesh.unit_square(2),
        compliance=data.compliance(), source=data.force, dirichlet=data.displacement,
        local_refinement=1, quadrature_order=8)
    print("Displacement L2 error:", stress.l2_error(data.displacement, 9))
    print("Full stress L2 error:", stress.stress_l2_error(data.stress, 9))


The mixed displacement is discontinuous. Profiles evaluate each fine-cell polynomial independently; separate segments preserve one-sided values. The local projection decomposition distinguishes approximation error from the difference between the solved displacement and its best local approximation.

In [ ]:
record = json.loads((ROOT / "examples/results/core-extensions/elasticity-projection.json").read_text())
for row in record["rows"]:
    print(row["resolution"], {name: float(np.linalg.norm(row[name])) for name in
          ("projection_l2", "discrete_l2", "total_l2")})
display(Image(filename=str(ROOT / "docs/figures/core-extensions/polygon-bdm2-displacement-profiles.png")))


In [ ]:
from pymhm import AffineMixedMesh, solve_darcy_hdiv3d

def pressure(x):
    """Quadratic exact pressure with a represented normal trace."""
    return np.sum(x*x, axis=1)

with threadpool_limits(1):
    mixed = solve_darcy_hdiv3d(AffineMixedMesh.unit_cube(), pressure_degree=2,
        normal_degree=2, trace_degree=2, local_refinement=1,
        source=-6.0, dirichlet=pressure, quadrature_order=8)
    assert mixed.errors(pressure, lambda x: -2*x, 9)["flux_l2"] < 1e-10
    print(mixed.errors(pressure, lambda x: -2*x, 9))


In [ ]:
from pymhm import TetraMesh, solve_mshho_3d
mesh = TetraMesh.unit_cube()
neumann = {int(f): -2*mesh.points[mesh.faces[f]].mean(axis=0)@mesh.normals[f]
           for f in mesh.boundary_faces}
with threadpool_limits(1):
    hho = solve_mshho_3d(mesh, source=-6.0, neumann=neumann, mean_pressure=1.0)
    assert hho.l2_error(pressure) < 2e-11
    assert hho.flux_l2_error(lambda x: -2*x) < 2e-10
    print("MsHHO pressure / raw flux errors:", hho.l2_error(pressure),
          hho.flux_l2_error(lambda x: -2*x))


In [ ]:
for suite in ("elasticity", "hdiv3d", "mshho3d"):
    record = json.loads((ROOT / f"examples/results/core-extensions/{suite}.json").read_text())
    print(suite, len(record["rows"]), "acquired cases")
    display(Image(filename=str(ROOT / f"docs/figures/core-extensions/{suite}-convergence.png")))


The high-order comparison separates local interior degree and skeletal normal degree. Macro equilibrium and small algebraic residuals do not establish approximation accuracy. The records report physical norms and independently selected error quadratures.